In [1]:
import sys
sys.path.append('../cace/')

import numpy as np
import torch
import torch.nn as nn
import logging
import datetime

import cace
from cace.representations import Cace
from cace.modules import CosineCutoff, MollifierCutoff, PolynomialCutoff
from cace.modules import BesselRBF, GaussianRBF, GaussianRBFCentered

from cace.models.atomistic import NeuralNetworkPotential
from cace.tasks.train import TrainingTask

torch.set_default_dtype(torch.float32)

cace.tools.setup_logger(level='INFO')
cutoff = 4.5

#val_ratio = float(sys.argv[1])
val_ratio = 0.1

Fourier_node = 21
save_folder = "/dssg/home/acct-matxzl/matxzl/Yajie/MDNN/cace-lr-fit-main/fit-electrolyte-B/loss_data/N_200_N_"+str(Fourier_node)+"_"
now = datetime.datetime.now()
time_name=now.strftime("%Y%m%d_%H%M%S")

print("reading data")
collection = cace.tasks.get_dataset_from_xyz(train_path='/dssg/home/acct-matxzl/matxzl/Yajie/MDNN/cace-lr-fit-main/fit-electrolyte-B/electrolyte_200.xyz',
                                 valid_fraction=val_ratio,
                                 seed=1,
                                 cutoff=cutoff,
                                 data_key={'energy': 'energy', 'forces':'forces'}, 
                                 atomic_energies={1: -0.1749365806299343, 8: -0.08746829031496617, 9: -4.620975436064299, 19: -4.620975436064285} # avg
                                 )
batch_size = 5

train_loader = cace.tasks.load_data_loader(collection=collection,
                              data_type='train',
                              batch_size=batch_size,
                              )

valid_loader = cace.tasks.load_data_loader(collection=collection,
                              data_type='valid',
                              batch_size=10,
                              )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = cace.tools.init_device(use_device)
print(f"device: {device}")

reading data
2025-03-23 16:59:45.327 INFO: Loaded 200 training configurations from '/dssg/home/acct-matxzl/matxzl/Yajie/MDNN/cace-lr-fit-main/fit-electrolyte-B/electrolyte_200.xyz'
2025-03-23 16:59:45.328 INFO: Using random 10.0% of training set for validation
device: cuda


In [2]:
print("building CACE representation")
radial_basis = BesselRBF(cutoff=cutoff, n_rbf=6, trainable=True)
#cutoff_fn = CosineCutoff(cutoff=cutoff)
cutoff_fn = PolynomialCutoff(cutoff=cutoff)

cace_representation = Cace(
    zs=[1, 8, 9, 19],
    n_atom_basis=4,
    embed_receiver_nodes=True,
    cutoff=cutoff,
    cutoff_fn=cutoff_fn,
    radial_basis=radial_basis,
    n_radial_basis=12,
    max_l=3,
    max_nu=3,
    num_message_passing=0,
    type_message_passing=['Bchi'],
    args_message_passing={'Bchi': {'shared_channels': False, 'shared_l': False}},
    #avg_num_neighbors=1,
    device=device,
    timeit=False
           )

cace_representation.to(device)
print(f"Representation: {cace_representation}")

atomwise = cace.modules.atomwise.Atomwise(n_layers=3,
                                         output_key='CACE_energy',
                                         n_hidden=[32,16],
                                         use_batchnorm=False,
                                         add_linear_nn=True)

forces = cace.modules.forces.Forces(energy_key='CACE_energy',
                                    forces_key='CACE_forces')

print("building CACE NNP")
cace_nnp_sr = NeuralNetworkPotential(
    input_modules=None,
    representation=cace_representation,
    output_modules=[atomwise, forces]
)

q = cace.modules.Atomwise(
    n_layers=3,
    n_hidden=[24,12],
    n_out=1,
    per_atom_output_key='q',
    output_key = 'tot_q',
    residual=False,
    add_linear_nn=True,
    bias=False)

ep = cace.modules.SOGPotential(NpointsMesh=Fourier_node,
                    feature_key='q',
                    output_key='SOG_potential',
                    remove_self_interaction=False,
                    aggregation_mode='sum',
                    Periodic = True)

forces_lr = cace.modules.Forces(energy_key='SOG_potential',
                                    forces_key='SOG_forces')

cace_nnp_lr = NeuralNetworkPotential(
    input_modules=None,
    representation=cace_representation,
    output_modules=[q, ep, forces_lr]
)

pot2 = {'CACE_energy': 'SOG_potential', 
        'CACE_forces': 'SOG_forces',
        'weight': 1
       }

pot1 = {'CACE_energy': 'CACE_energy', 
        'CACE_forces': 'CACE_forces',
       }

cace_nnp = cace.models.CombinePotential([cace_nnp_sr, cace_nnp_lr], [pot1,pot2])
#cace_nnp = cace.models.CombinePotential([cace_nnp_sr], [pot1])
cace_nnp.to(device)


building CACE representation
Representation: Cace(
  (node_onehot): NodeEncoder(num_classes=4)
  (node_embedding_sender): NodeEmbedding(num_classes=4, embedding_dim=4)
  (node_embedding_receiver): NodeEmbedding(num_classes=4, embedding_dim=4)
  (edge_coding): EdgeEncoder(directed=True)
  (radial_basis): BesselRBF(cutoff=4.5, n_rbf=6, trainable=True)
  (cutoff_fn): PolynomialCutoff(p=6.0, cutoff=4.5)
  (angular_basis): AngularComponent(l_max=3)
  (radial_transform): SharedRadialLinearTransform(
    (weights): ParameterList(
        (0): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
        (1): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
        (2): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
        (3): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
    )
  )
  (symmetrizer): Symmetrizer()
  (message_passing_list): ModuleList()
)
building CACE NNP


CombinePotential(
  (models): ModuleList(
    (0): NeuralNetworkPotential(
      (postprocessors): ModuleList()
      (representation): Cace(
        (node_onehot): NodeEncoder(num_classes=4)
        (node_embedding_sender): NodeEmbedding(num_classes=4, embedding_dim=4)
        (node_embedding_receiver): NodeEmbedding(num_classes=4, embedding_dim=4)
        (edge_coding): EdgeEncoder(directed=True)
        (radial_basis): BesselRBF(cutoff=4.5, n_rbf=6, trainable=True)
        (cutoff_fn): PolynomialCutoff(p=6.0, cutoff=4.5)
        (angular_basis): AngularComponent(l_max=3)
        (radial_transform): SharedRadialLinearTransform(
          (weights): ParameterList(
              (0): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (1): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (2): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (3): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)

In [3]:
print(f"First train loop:")
energy_loss = cace.tasks.GetLoss(
    target_name='energy',
    predict_name='CACE_energy',
    loss_fn=torch.nn.MSELoss(),
    loss_weight=0.0#0.1
)

force_loss = cace.tasks.GetLoss(
    target_name='forces',
    predict_name='CACE_forces',
    loss_fn=torch.nn.MSELoss(),
    loss_weight=1.0#1000
)

from cace.tools import Metrics

e_metric = Metrics(
    target_name='energy',
    predict_name='CACE_energy',
    name='e/atom',
    per_atom=True
)

f_metric = Metrics(
    target_name='forces',
    predict_name='CACE_forces',
    name='f'
)

# Example usage
print("creating training task")

boost = int((1./(1.-val_ratio))**0.5)

optimizer_args = {'lr': 1e-2, 'betas': (0.99, 0.999)}  
scheduler_args = {'step_size': 20*boost, 'gamma': 0.5}

for i in range(5):
    task = TrainingTask(
        model=cace_nnp,
        losses=[energy_loss, force_loss],
        metrics=[e_metric, f_metric],
        device=device,
        optimizer_args=optimizer_args,
        scheduler_cls=torch.optim.lr_scheduler.StepLR,
        scheduler_args=scheduler_args,
        max_grad_norm=10,
        ema=False, #True,
        ema_start=10,
        warmup_steps=5,
        save_folder = save_folder,
        time_name = time_name
    )

    print("training")
    task.fit(train_loader, valid_loader, epochs=40*boost, screen_nan=False, val_stride=10)

task.save_model(save_folder+'electrolyte-model.pth')
cace_nnp.to(device)


First train loop:
creating training task
training
Train loss:0.7292541861534119
shift: Parameter containing:
tensor([-1.0000, -0.8182, -0.6364, -0.4545, -0.2727, -0.0909,  0.0909,  0.2727,
         0.4545,  0.6364,  0.8182,  1.0000], device='cuda:0',
       requires_grad=True)
weight: Parameter containing:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], device='cuda:0',
       requires_grad=True)
Train loss:0.7186775803565979
shift: Parameter containing:
tensor([-1.0020, -0.8202, -0.6384, -0.4565, -0.2747, -0.0929,  0.0889,  0.2707,
         0.4525,  0.6344,  0.8162,  0.9980], device='cuda:0',
       requires_grad=True)
weight: Parameter containing:
tensor([1.0020, 1.0020, 1.0020, 1.0020, 1.0020, 1.0020, 1.0020, 1.0020, 1.0020,
        1.0020, 1.0020, 1.0020], device='cuda:0', requires_grad=True)
Train loss:0.7419596910476685
Train loss:0.7140175104141235
shift: Parameter containing:
tensor([-1.0059, -0.8241, -0.6423, -0.4605, -0.2787, -0.0969,  0.0849,  0.2667,
         0.448

CombinePotential(
  (models): ModuleList(
    (0): NeuralNetworkPotential(
      (postprocessors): ModuleList()
      (representation): Cace(
        (node_onehot): NodeEncoder(num_classes=4)
        (node_embedding_sender): NodeEmbedding(num_classes=4, embedding_dim=4)
        (node_embedding_receiver): NodeEmbedding(num_classes=4, embedding_dim=4)
        (edge_coding): EdgeEncoder(directed=True)
        (radial_basis): BesselRBF(cutoff=4.5, n_rbf=6, trainable=True)
        (cutoff_fn): PolynomialCutoff(p=6.0, cutoff=4.5)
        (angular_basis): AngularComponent(l_max=3)
        (radial_transform): SharedRadialLinearTransform(
          (weights): ParameterList(
              (0): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (1): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (2): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (3): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)

In [7]:
optimizer_args = {'lr': 1e-3, 'betas': (0.99, 0.999)}  
scheduler_args = {'step_size': 20*boost, 'gamma': 0.95}

print(f"Second train loop:")
energy_loss2 = cace.tasks.GetLoss(
    target_name='energy',
    predict_name='CACE_energy',
    loss_fn=torch.nn.MSELoss(),
    loss_weight=0.001
)

for i in range(5):
    task = TrainingTask(
        model=cace_nnp,
        losses=[energy_loss2, force_loss],
        metrics=[e_metric, f_metric],
        device=device,
        optimizer_args=optimizer_args,
        scheduler_cls=torch.optim.lr_scheduler.StepLR,
        scheduler_args=scheduler_args,
        max_grad_norm=10,
        ema=False, #True,
        ema_start=10,
        warmup_steps=5,
        save_folder = save_folder,
        time_name = time_name
    )
    task.fit(train_loader, valid_loader, epochs=50*boost, screen_nan=False, val_stride=10)

task.save_model(save_folder+'electrolyte-model-2.pth')
cace_nnp.to(device)


Second train loop:
Train loss:0.012551727704703808
shift: Parameter containing:
tensor([0.0562, 0.8113, 1.1294, 0.6598, 0.6346, 0.6312, 1.0942, 1.0357, 0.6495,
        0.9513, 1.0076, 1.0157], device='cuda:0', requires_grad=True)
weight: Parameter containing:
tensor([-0.2033, -0.5586, -0.0533,  0.8267,  1.4396,  1.8954, -1.2516, -1.3510,
         0.7147, -1.2465, -1.9181, -1.8245], device='cuda:0',
       requires_grad=True)
Train loss:0.011709233745932579
shift: Parameter containing:
tensor([0.0564, 0.8115, 1.1296, 0.6596, 0.6344, 0.6310, 1.0944, 1.0359, 0.6493,
        0.9515, 1.0078, 1.0159], device='cuda:0', requires_grad=True)
weight: Parameter containing:
tensor([-0.2031, -0.5584, -0.0531,  0.8269,  1.4398,  1.8956, -1.2514, -1.3508,
         0.7149, -1.2463, -1.9179, -1.8243], device='cuda:0',
       requires_grad=True)
Train loss:0.01202996913343668
Train loss:0.010465501807630062
shift: Parameter containing:
tensor([0.0564, 0.8117, 1.1296, 0.6595, 0.6342, 0.6309, 1.0945, 1.036

CombinePotential(
  (models): ModuleList(
    (0): NeuralNetworkPotential(
      (postprocessors): ModuleList()
      (representation): Cace(
        (node_onehot): NodeEncoder(num_classes=4)
        (node_embedding_sender): NodeEmbedding(num_classes=4, embedding_dim=4)
        (node_embedding_receiver): NodeEmbedding(num_classes=4, embedding_dim=4)
        (edge_coding): EdgeEncoder(directed=True)
        (radial_basis): BesselRBF(cutoff=4.5, n_rbf=6, trainable=True)
        (cutoff_fn): PolynomialCutoff(p=6.0, cutoff=4.5)
        (angular_basis): AngularComponent(l_max=3)
        (radial_transform): SharedRadialLinearTransform(
          (weights): ParameterList(
              (0): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (1): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (2): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)]
              (3): Parameter containing: [torch.float32 of size 6x12x16 (cuda:0)

In [11]:
optimizer_args = {'lr': 5e-4, 'betas': (0.99, 0.999)}  
scheduler_args = {'step_size': 20*boost, 'gamma': 0.95}

energy_loss3 = cace.tasks.GetLoss(
    target_name='energy',
    predict_name='CACE_energy',
    loss_fn=torch.nn.MSELoss(),
    loss_weight=0.0001 
)
print(f"Third train loop:")
for i in range(20):
    task = TrainingTask(
        model=cace_nnp,
        losses=[energy_loss3, force_loss],
        metrics=[e_metric, f_metric],
        device=device,
        optimizer_args=optimizer_args,
        scheduler_cls=torch.optim.lr_scheduler.StepLR,
        scheduler_args=scheduler_args,
        max_grad_norm=10,
        ema=False, #True,
        ema_start=10,
        warmup_steps=5,
        save_folder = save_folder,
        time_name = time_name
    )
    task.fit(train_loader, valid_loader, epochs=50*boost, screen_nan=False, val_stride=10)

task.save_model(save_folder+'electrolyte-model-3.pth')

Third train loop:
Train loss:0.00016192668408621103
shift: Parameter containing:
tensor([0.2217, 1.4283, 1.4297, 0.8990, 0.8420, 0.3436, 1.5824, 1.5879, 0.8953,
        1.4489, 3.1744, 3.1636], device='cuda:0', requires_grad=True)
weight: Parameter containing:
tensor([ 0.0893,  4.8112,  4.7628,  2.1602,  1.6431,  1.9017,  4.2987,  4.2592,
         1.6937,  4.5790, -0.4594, -0.3595], device='cuda:0',
       requires_grad=True)
Train loss:0.00024106691125780344
Train loss:0.00011394302418921143
shift: Parameter containing:
tensor([0.2218, 1.4284, 1.4298, 0.8988, 0.8420, 0.3436, 1.5825, 1.5880, 0.8953,
        1.4490, 3.1744, 3.1636], device='cuda:0', requires_grad=True)
weight: Parameter containing:
tensor([ 0.0892,  4.8112,  4.7627,  2.1601,  1.6431,  1.9017,  4.2987,  4.2591,
         1.6937,  4.5789, -0.4594, -0.3595], device='cuda:0',
       requires_grad=True)
Train loss:0.00018086933414451778
Train loss:0.000165574747370556
Train loss:0.00017740993644110858
Train loss:0.00013337668

In [ ]:
print(f"Finished")

trainable_params = sum(p.numel() for p in cace_nnp.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params}")